In [ ]:
import os
from kaggle_secrets import UserSecretsClient

In [13]:
customer_email = """
Arrr, I be fuming that me blender lid \
flew off and splattered me kitchen walls \
with smoothie! And to make matters worse,\
the warranty don't cover the cost of \
cleaning up me kitchen. I need yer help \
right now, matey!
"""

In [14]:
style = """American English \
in a calm and respectful tone
"""

prompt = f"""Translate the text \
that is delimited by triple backticks 
into a style that is {style}.
text: ```{customer_email}```
"""
print(prompt)

Translate the text that is delimited by triple backticks 
into a style that is American English in a calm and respectful tone
.
text: ```
Arrr, I be fuming that me blender lid flew off and splattered me kitchen walls with smoothie! And to make matters worse,the warranty don't cover the cost of cleaning up me kitchen. I need yer help right now, matey!
```



# Prompt Templete using LangChain 

In [19]:
pip install -U langchain-google-genai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 724.4/724.4 kB 14.0 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 236.5/236.5 kB 13.2 MB/s eta 0:00:00
  Attempting uninstall: google-auth
    Found existing installation: google-auth 2.43.0
    Uninstalling google-auth-2.43.0:
      Successfully uninstalled google-auth-2.43.0
  Attempting uninstall: google-genai
    Found existing installation: google-genai 1.55.0
    Uninstalling google-genai-1.55.0:
      Successfully uninstalled google-genai-1.55.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.31.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.21.0 requires google-cloud-bigquery-s

In [ ]:
llm_model = "gemini-2.5-flash"

In [21]:
from google import genai

In [26]:
from langchain_google_genai import ChatGoogleGenerativeAI 

In [ ]:
os.environ["GOOGLE_API_KEY"] = user_secrets.get_secret("Gemini_API")
chat = ChatGoogleGenerativeAI(
    model=llm_model,
    temperature=0
)

In [36]:
msg = chat.invoke(prompt)


In [37]:
msg.content

"I'm writing to you today because I've encountered a rather frustrating situation. My blender lid unexpectedly came off, causing smoothie to splatter across my kitchen walls. To add to my concern, I've learned that the warranty does not cover the cost of cleaning up my kitchen. I would be very grateful for your assistance with this matter."

In [38]:
template_string = """Translate the text \
that is delimited by triple backticks \
into a style that is {style}. \
text: ```{text}```
"""

In [41]:
from langchain_core.prompts import ChatPromptTemplate

In [43]:
prompt_template = ChatPromptTemplate.from_template(template_string)

In [47]:
prompt_template.messages[0].prompt

PromptTemplate(input_variables=['style', 'text'], input_types={}, partial_variables={}, template='Translate the text that is delimited by triple backticks into a style that is {style}. text: ```{text}```\n')

In [49]:
prompt_template.messages[0].prompt.input_variables

['style', 'text']

In [50]:
customer_style = """American English \
in a calm and respectful tone
"""

customer_email = """
Arrr, I be fuming that me blender lid \
flew off and splattered me kitchen walls \
with smoothie! And to make matters worse, \
the warranty don't cover the cost of \
cleaning up me kitchen. I need yer help \
right now, matey!
"""

In [55]:
customer_messages= prompt_template.format_messages(
                    style=customer_style,
                    text=customer_email
)

In [56]:
print(type(customer_messages))
print(type(customer_messages[0]))

<class 'list'>
<class 'langchain_core.messages.human.HumanMessage'>


In [57]:
customer_response = chat.invoke(customer_messages)

In [59]:
print(customer_response.content)

I'm quite upset because my blender lid came off and splattered smoothie all over my kitchen walls. To make matters worse, the warranty doesn't cover the cost of cleaning my kitchen. I would appreciate your assistance with this matter.


In [60]:
template = """
Determine if the student's solution is correct or not.

Question:
{question}

Student's Solution:
{solution}
"""

In [61]:
prompt_template = ChatPromptTemplate.from_template(template)

In [63]:
prompt_template.messages[0].prompt.input_variables

['question', 'solution']

In [64]:
question = """I'm building a solar power installation...
- Land costs $100 / square foot
- I can buy solar panels for $250 / square foot
- Maintenance: $100k per year + $10 / square foot
What is the total cost for the first year as a function of x?"""

In [65]:
solution = """Let x be the size...
Land: 100x
Panels: 250x
Maintenance: 100,000 + 100x
Total: 450x + 100,000"""

In [67]:
student_correctness = prompt_template.format_messages(
                    question=question,
                    solution=solution
)

In [68]:
response = chat.invoke(student_correctness)

In [71]:
response.content

'The student\'s solution is incorrect.\n\nHere\'s a breakdown:\n\n1.  **Land Cost:** Correct. $100 \\times x = 100x$\n2.  **Panels Cost:** Correct. $250 \\times x = 250x$\n3.  **Maintenance Cost:** **Incorrect.**\n    *   The problem states: "$100k per year + $10 / square foot"\n    *   This should be: $100,000 + 10x$\n    *   The student wrote: $100,000 + 100x$ (They used $100/sq ft instead of $10/sq ft for the variable part of maintenance).\n\n**Correct Solution:**\n\n*   Land: $100x$\n*   Panels: $250x$\n*   Maintenance: $100,000 + 10x$\n\n**Total Cost:**\n$100x + 250x + (100,000 + 10x)$\n$= (100x + 250x + 10x) + 100,000$\n$= 360x + 100,000$'